# MGT-432 Group Assignment · Team notebook

This notebook trains your team's model on `train.csv` and the small labelled `adaptation.csv`, saves it as `team_model.pkl`, and uses the saved model to write two files:

- `predictions.csv`: a revenue forecast for every test film
- `valuations.csv`: your auction sheet, one row per auction lot with whatever columns you want to bid with

**What we do with it.** After the deadline we run section 6 with your `team_model.pkl`, using the hidden test and auction files. We score only `predicted_revenue`. We print the complete `valuations.csv` and give it to your team before the auction. If your saved model cannot be loaded on our computer, we run the whole notebook instead.

**What you do**

1. Improve the cells marked ✏️ (sections 0, 2 to 5). Leave sections 1 and 6 as they are.
2. List every library you import in `requirements.txt` (section 0).
3. Run all cells. Section 5 saves your model as `team_model.pkl` and checks `requirements.txt`; section 6 loads the model, writes the two files and prints two ✅ lines.
4. Check that section 6 works on its own: **restart the kernel** (clearing all outputs is fine), then run only the two cells of section 6. They must print the two ✅ lines again.
5. Upload three files per team: this `.ipynb`, `team_model.pkl` and `requirements.txt`.

## 0 · Files and libraries ✏️

`requirements.txt` must list every library this notebook imports, one per line, with the version you use (for example `lightgbm==4.6.0`). We install exactly these libraries to load your model. The check at the end of section 5 tells you what to add or change.

To install the listed libraries, for example on Google Colab or a new computer, remove the `#` in front of `!pip install` and run the cell.

In [1]:
# !pip install -r requirements.txt

Change the file names only if the data are somewhere else. On Google Colab, upload `train.csv`, `adaptation.csv`, `sample_features.csv` and `requirements.txt`, and remove `../` below.

In [2]:
import os
print(os.getcwd())

/Users/alegonzalez/PycharmProjects/DSB_assignment/starter_kit


In [3]:
TRAIN_FILE = "../train.csv"
ADAPTATION_FILE = "../adaptation.csv"

## 1 · Load the training data (do not edit)

Three columns of `train.csv` are not in the test and auction files: the target `revenue` and the identifiers `movie_id` and `title`. Your features cannot use them.

`adaptation.csv` contains 396 labelled, anonymous films from the same out-of-sample setting as Hidden Test B, whose distribution may differ from `train.csv`. It has the deployment features plus `adaptation_id` and `revenue`. Do not use either ID as a feature.

In [4]:
import numpy as np
import pandas as pd

train = pd.read_csv(TRAIN_FILE)
adaptation = pd.read_csv(ADAPTATION_FILE)

print(len(train), "training films")
print(len(adaptation), "adaptation films")
train.head()

3943 training films
396 adaptation films


,movie_id,title,release_year,release_quarter,runtime,budget,original_language,primary_genre,genres,n_genres,production_countries,n_companies,company_ids,n_cast,n_crew,in_collection,keyword_ids,cast_ids,director_id,revenue
0,M3ddf258d,Jumanji,1995,4,105,65000000,en,Adventure,Adventure|Fantasy|Family,3,US,3,C6dfc8237|C6bee1098|Ceafcc337,26,16,False,Kcba84b9c|K6b9ad492|K31b25f7e|Kacea562a|K7b321...,P8b29037b|Pcc7ebcea|Pfe6d4c3d|P45aad4a2|P4c455563,Pa589cb62,262797249.0
1,M862e1fd4,Heat,1995,4,170,60000000,en,Action,Action|Crime|Drama|Thriller,4,US,3,Cc2c73c36|C4afff6ec|C3e7bdc8f,65,71,False,Kc5b65b8c|Kb1aa9b34|Kd95b93ea|K443b5d4d|K00ee1...,P13753a9d|Pd9604827|Pcfb78473|Pb3cd0383|P575dda13,Paa8ef28b,187436818.0
2,M4617e480,Sudden Death,1995,4,105,35000000,en,Action,Action|Adventure|Thriller,3,US,3,Cd7a83b16|C0866a564|C79ac32e3,6,9,False,K4d2febdc|K8b8060e4|K18f7c353|K4f7f609b,P7ff40d0f|P387c2084|P972106f7|P12aa5436|P22cc2e84,Pc2c37f32,64350171.0
3,Medf94ac1,The American President,1995,4,105,62000000,en,Comedy,Comedy|Drama|Romance,3,US,2,Cf4478484|Ce4e76224,18,6,False,Kfb526a76|K2b746c3e|Kfa895068|K1355f779|Kd57fb469,P9485f066|Peb51c4ea|P977f7843|P5768eddf|Pd3b64bd6,P4bc483e6,107879496.0
4,Mfec91032,Nixon,1995,4,190,44000000,en,History,History|Drama,2,US,2,C2b01c913|C4ca4dc32,34,8,False,K2b746c3e|K3ccb5600|K85e45b97|K4385ab38|K8beac...,Pdff627c7|P7b4520e9|P387c2084|Pceb538c7|P09ae177a,Pc20617a1,13681765.0


In [5]:
print(train.columns)

Index(['movie_id', 'title', 'release_year', 'release_quarter', 'runtime',
       'budget', 'original_language', 'primary_genre', 'genres', 'n_genres',
       'production_countries', 'n_companies', 'company_ids', 'n_cast',
       'n_crew', 'in_collection', 'keyword_ids', 'cast_ids', 'director_id',
       'revenue'],
      dtype='object')


## 2 · Features ✏️

`make_features` turns a table of films into the numbers the model learns from. The same function is applied to the training, adaptation, hidden test and auction films. It must ignore targets, titles and row identifiers.

Features are built in two steps:

1. **`make_features(films)`** cleans each film on its own row (logs, counts, splitting the `|` lists). Nothing is learned from the data, so it gives the same result whatever file it is applied to.
2. **`make_preprocessor()`** returns the encoders that *learn* from the training films (which categories are frequent, the track record of each person). It goes **inside the model pipeline** in section 3, so it is refitted on exactly the films each model is trained on, and never sees the validation films.

How each column is handled, based on what we saw in the data:

| Columns | What the data shows | Treatment |
|---|---|---|
| `budget` | from \$10k to \$400M, very skewed; no zeros or missing values | `log10(budget)` |
| `runtime`, `release_year`, `release_quarter`, `n_genres` | clean, no missing values | used as they are |
| `n_cast`, `n_crew`, `n_companies` | skewed. The catalogue lists far bigger casts and crews for recent films (median crew about 19 before 2015 and above 80 after 2018), so the raw counts drift over time | `log1p`, plus cast and crew size **relative to the median film of the same year** |
| `in_collection` | bool; franchise films earn about 4-10× more | 0/1 |
| `original_language`, `primary_genre` | 36 and 20 values, every adaptation value is seen in train | one-hot; values with fewer than 20 films grouped as "infrequent" |
| `genres`, `production_countries` | lists of about 20 genres and 70 countries (26% of films have several countries, so the raw string is not a category) | one column per item seen in 20 or more films, plus the number of countries |
| `director_id`, `cast_ids`, `company_ids`, `keyword_ids` | thousands of IDs, most appearing in very few films; about 40-75% of adaptation IDs also appear in train | **track record**: smoothed average log-revenue of the ID's other films (mean and max over the film's IDs) and how many films those IDs have, plus the number of keywords |
| missing `company_ids`, `keyword_ids`, `production_countries` | a few hundred films | treated as an empty list |

In [6]:
from collections import Counter, defaultdict

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder


def as_list(x):
    if isinstance(x, (list, tuple, set)):
        return list(x)
    if isinstance(x, str):
        return [t.strip() for t in x.split("|") if t.strip()]
    return []


numeric_columns = [
    "release_year",
    "release_quarter",
    "runtime",
    "log_budget",
    "n_genres",
    "log_n_companies",
    "log_n_cast",
    "log_n_crew",
    "in_collection",
    "n_keywords",
    "n_countries",
]

categorical_columns = ["original_language", "primary_genre"]

multi_hot_columns = ["genres", "production_countries"]

track_record_columns = ["director_id", "cast_ids", "company_ids", "keyword_ids"]


def make_features(films):
    f = pd.DataFrame(index=films.index)
    f["release_year"] = films["release_year"]
    f["release_quarter"] = films["release_quarter"]
    f["runtime"] = films["runtime"]
    f["log_budget"] = np.log10(films["budget"].clip(lower=1000))
    f["n_genres"] = films["n_genres"]
    f["log_n_companies"] = np.log1p(films["n_companies"])
    f["log_n_cast"] = np.log1p(films["n_cast"])
    f["log_n_crew"] = np.log1p(films["n_crew"])
    f["in_collection"] = films["in_collection"].fillna(False).astype(int)
    f["n_keywords"] = films["keyword_ids"].map(lambda x: len(as_list(x)))
    f["n_countries"] = films["production_countries"].map(lambda x: len(as_list(x)))
    for col in categorical_columns:
        f[col] = films[col].fillna("unknown")
    for col in multi_hot_columns + track_record_columns:
        f[col] = films[col].map(as_list)  # director_id becomes a one-item list
    return f


make_features(train).head()

,release_year,release_quarter,runtime,log_budget,n_genres,log_n_companies,log_n_cast,log_n_crew,in_collection,n_keywords,n_countries,original_language,primary_genre,genres,production_countries,director_id,cast_ids,company_ids,keyword_ids
0,1995,4,105,7.812913,3,1.386294,3.295837,2.833213,0,6,1,en,Adventure,"[Adventure, Fantasy, Family]",[US],[Pa589cb62],"[P8b29037b, Pcc7ebcea, Pfe6d4c3d, P45aad4a2, P...","[C6dfc8237, C6bee1098, Ceafcc337]","[Kcba84b9c, K6b9ad492, K31b25f7e, Kacea562a, K..."
1,1995,4,170,7.778151,4,1.386294,4.189655,4.276666,0,8,1,en,Action,"[Action, Crime, Drama, Thriller]",[US],[Paa8ef28b],"[P13753a9d, Pd9604827, Pcfb78473, Pb3cd0383, P...","[Cc2c73c36, C4afff6ec, C3e7bdc8f]","[Kc5b65b8c, Kb1aa9b34, Kd95b93ea, K443b5d4d, K..."
2,1995,4,105,7.544068,3,1.386294,1.945910,2.302585,0,4,1,en,Action,"[Action, Adventure, Thriller]",[US],[Pc2c37f32],"[P7ff40d0f, P387c2084, P972106f7, P12aa5436, P...","[Cd7a83b16, C0866a564, C79ac32e3]","[K4d2febdc, K8b8060e4, K18f7c353, K4f7f609b]"
3,1995,4,105,7.792392,3,1.098612,2.944439,1.945910,0,5,1,en,Comedy,"[Comedy, Drama, Romance]",[US],[P4bc483e6],"[P9485f066, Peb51c4ea, P977f7843, P5768eddf, P...","[Cf4478484, Ce4e76224]","[Kfb526a76, K2b746c3e, Kfa895068, K1355f779, K..."
4,1995,4,190,7.643453,2,1.098612,3.555348,2.197225,0,6,1,en,History,"[History, Drama]",[US],[Pc20617a1],"[Pdff627c7, P7b4520e9, P387c2084, Pceb538c7, P...","[C2b01c913, C4ca4dc32]","[K2b746c3e, K3ccb5600, K85e45b97, K4385ab38, K..."


In [7]:
class ListEncoder(BaseEstimator, TransformerMixin):
    """One 0/1 column per item seen in at least `min_count` training films."""

    def __init__(self, min_count=20):
        self.min_count = min_count

    def fit(self, X, y=None):
        self.column_ = X.columns[0]
        counts = Counter(item for items in X.iloc[:, 0] for item in items)
        self.classes_ = sorted(k for k, n in counts.items() if n >= self.min_count)
        self.mlb_ = MultiLabelBinarizer(classes=self.classes_).fit([])
        return self

    def transform(self, X):
        known = set(self.classes_)
        encoded = self.mlb_.transform([[i for i in items if i in known] for items in X.iloc[:, 0]])
        return pd.DataFrame(encoded, index=X.index, columns=self.get_feature_names_out())

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{self.column_}_{c}" for c in self.classes_], dtype=object)


class YearRelative(BaseEstimator, TransformerMixin):
    """Cast and crew size minus the median of the training films released the same year.

    Years not seen in training use the closest year that was.
    """

    def fit(self, X, y=None):
        self.columns_ = [c for c in X.columns if c != "release_year"]
        self.medians_ = X.groupby("release_year")[self.columns_].median()
        return self

    def transform(self, X):
        years = self.medians_.index.to_numpy()
        closest = years[np.abs(X["release_year"].to_numpy()[:, None] - years).argmin(axis=1)]
        relative = X[self.columns_].to_numpy() - self.medians_.loc[closest].to_numpy()
        return pd.DataFrame(relative, index=X.index, columns=self.get_feature_names_out())

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{c}_vs_year" for c in self.columns_], dtype=object)


class TrackRecord(BaseEstimator, TransformerMixin):
    """Track record of the IDs of a film (director, actors, companies or keywords).

    Each ID gets the smoothed average log10-revenue of its training films:
        (sum of log-revenues + smoothing * overall mean) / (number of films + smoothing)
    so an ID with one film stays close to the overall mean. A film gets the mean and
    the max over its known IDs (the overall mean if none is known), and the log of the
    total number of films of its IDs.

    On the training films, fit_transform uses out-of-fold values: a film's own revenue
    never describes itself, otherwise the model would over-trust these columns.
    """

    def __init__(self, smoothing=5, n_splits=5, random_state=432):
        self.smoothing = smoothing
        self.n_splits = n_splits
        self.random_state = random_state

    def _learn(self, lists, y):
        sums, counts = defaultdict(float), defaultdict(int)
        for items, target in zip(lists, y):
            for item in set(items):
                sums[item] += target
                counts[item] += 1
        prior = float(np.mean(y))
        table = {k: (sums[k] + self.smoothing * prior) / (counts[k] + self.smoothing) for k in sums}
        return table, counts, prior

    def _apply(self, lists, table, counts, prior):
        out = np.empty((len(lists), 3))
        for r, items in enumerate(lists):
            known = [table[i] for i in items if i in table]
            out[r, 0] = np.mean(known) if known else prior
            out[r, 1] = np.max(known) if known else prior
            out[r, 2] = np.log1p(sum(counts[i] for i in items if i in counts))
        return out

    def _frame(self, values, X):
        return pd.DataFrame(values, index=X.index, columns=self.get_feature_names_out())

    def fit(self, X, y):
        self.column_ = X.columns[0]
        self.table_, self.counts_, self.prior_ = self._learn(list(X.iloc[:, 0]), np.asarray(y, dtype=float))
        return self

    def transform(self, X):
        return self._frame(self._apply(list(X.iloc[:, 0]), self.table_, self.counts_, self.prior_), X)

    def fit_transform(self, X, y):
        lists, y = list(X.iloc[:, 0]), np.asarray(y, dtype=float)
        out = np.empty((len(lists), 3))
        folds = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for fit_idx, out_idx in folds.split(lists):
            table, counts, prior = self._learn([lists[i] for i in fit_idx], y[fit_idx])
            out[out_idx] = self._apply([lists[i] for i in out_idx], table, counts, prior)
        self.fit(X, y)
        return self._frame(out, X)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{self.column_}_mean", f"{self.column_}_max", f"{self.column_}_log_n_films"], dtype=object)

In [8]:
def make_preprocessor():
    return ColumnTransformer(
        [
            ("num", "passthrough", numeric_columns),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False),
             categorical_columns),
            ("size", YearRelative(), ["release_year", "log_n_cast", "log_n_crew"]),
        ]
        + [(col, ListEncoder(min_count=20), [col]) for col in multi_hot_columns]
        + [(col, TrackRecord(), [col]) for col in track_record_columns],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


# Preview only: in section 3 the preprocessor is fitted inside the model pipeline.
# preview = make_preprocessor().fit_transform(make_features(train), np.log10(train["revenue"]))
# print(preview.shape[1], "features")
# preview.head()



## 3 · Model ✏️

The model learns `log10(revenue)`: revenues range from thousands to billions of dollars, and the score (log-MAE) measures errors on this log scale. Fix `random_state` so that the model is the same every time you run the notebook.

The model is a pipeline of two steps, so that everything learned from data is refitted on exactly the films the model is trained on:

1. **`make_preprocessor()`** from section 2 turns the cleaned films into the numeric feature matrix (one-hot, multi-hot, year-relative sizes, out-of-fold track records).
2. **`HistGradientBoostingRegressor`**, scikit-learn's gradient-boosted trees:
   - It captures non-linear effects and interactions (e.g. budget matters more for some genres), which a linear regression cannot. Trees also need no scaling of the features.
   - `loss="absolute_error"` makes the trees minimise the absolute error on the log scale, which is exactly what log-MAE measures, and makes them less sensitive to the few films with extreme revenues.
   - A small learning rate with many trees, leaves of at least 20 films and some L2 regularisation keep it from overfitting 4,300 films.

In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def make_model():
    return make_pipeline(
        make_preprocessor(),
        HistGradientBoostingRegressor(
            loss="absolute_error",
            learning_rate=0.05,
            max_iter=500,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=432,
        ),
    )

## 4 · Validate ✏️

The hidden test has two parts, scored separately: Hidden Test A comes from the same distribution as `train.csv`; Hidden Test B is out-of-sample, and its distribution may differ. The 396 labelled adaptation films come from the same setting as Hidden Test B. The simple check below holds out about 20% of them, trains on `train.csv` plus the other 80%, and evaluates on the held-out adaptation films. You may replace it with cross-validation.

In [10]:
def log_mae(actual, forecast):
    return np.mean(np.abs(np.log10(forecast) - np.log10(actual)))

adapt_valid = adaptation.groupby("release_year", group_keys=False).sample(frac=0.20, random_state=432)
adapt_fit = adaptation.drop(index=adapt_valid.index)
development = pd.concat([train, adapt_fit], ignore_index=True, sort=False)

check_model = make_model().fit(make_features(development), np.log10(development["revenue"]))
forecast = 10 ** check_model.predict(make_features(adapt_valid))
median_guess = np.full(len(adapt_valid), development["revenue"].median())

print(f"Trained on {len(train)} training + {len(adapt_fit)} adaptation films; checked on {len(adapt_valid)} held-out adaptation films")
print(f"  your model          log-MAE = {log_mae(adapt_valid['revenue'], forecast):.3f}")
print(f"  training median     log-MAE = {log_mae(adapt_valid['revenue'], median_guess):.3f}  (lower is better)")

Trained on 3943 training + 316 adaptation films; checked on 80 held-out adaptation films
  your model          log-MAE = 0.402
  training median     log-MAE = 0.798  (lower is better)


## 5 · Train and save the final model ✏️

After validation, train on all training and adaptation films. Then define two functions. Each returns one result per film, in the same row order as `films`:

- `predict_revenue(films)` returns one positive revenue forecast in USD per film. This is what we score on the hidden test.
- `auction_sheet(films)` returns a DataFrame with whatever columns you want on your printed auction sheet, for example a valuation, an 80% interval, a maximum bid or a short note. It is not scored. Keep it concise so that the printout stays readable.

In [11]:
final_train = pd.concat([train, adaptation], ignore_index=True, sort=False)
model = make_model().fit(make_features(final_train), np.log10(final_train["revenue"]))

def predict_revenue(films):
    return 10 ** model.predict(make_features(films))

def auction_sheet(films):
    return pd.DataFrame({"valuation": predict_revenue(films)})

The next two cells save `predict_revenue` and `auction_sheet` in `team_model.pkl`, together with everything they use (your model, `make_features` and any lookup tables), and check that `requirements.txt` lists every library you use. Run them again whenever you change sections 2 to 5. Do not edit them.

In [12]:
#Do not edit it
import gzip, os, pickle, platform, sklearn
try:
    import cloudpickle                          # listed directly in requirements.txt
except ImportError:
    from joblib.externals import cloudpickle   # fallback for an older starter environment

with gzip.open("team_model.pkl", "wb") as f:
    pickle.dump({"python": platform.python_version(), "scikit-learn": sklearn.__version__}, f)
    cloudpickle.dump({"predict_revenue": predict_revenue, "auction_sheet": auction_sheet}, f)
print(f"✅ saved team_model.pkl ({os.path.getsize('team_model.pkl') / 1e6:.1f} MB)")

✅ saved team_model.pkl (1.0 MB)


In [13]:
#Do not edit it
import importlib.metadata, re, types

listed = {}
try:
    for line in open("requirements.txt"):
        line = line.split("#")[0].strip()
        if line:
            name = re.split(r"[<>=!~;\[ ]", line)[0].lower().replace("_", "-")
            listed[name] = line.split("==")[1].strip() if "==" in line else ""
except FileNotFoundError:
    print("⚠ requirements.txt not found: put it next to this notebook")

dist_of = importlib.metadata.packages_distributions()
used = set()
for obj in list(globals().values()):
    module = obj.__name__ if isinstance(obj, types.ModuleType) else getattr(obj, "__module__", None)
    if isinstance(module, str) and len(dist_of.get(module.split(".")[0], [])) == 1:
        used.add(dist_of[module.split(".")[0]][0])

missing = []
for dist in sorted(used):
    name, version = dist.lower().replace("_", "-"), importlib.metadata.version(dist)
    if name not in {"ipython", "ipykernel"} and listed.get(name) != version:
        missing.append(f"{name}=={version}")
if missing:
    print("⚠ Update requirements.txt so that it contains these lines:", *missing, sep="\n    ")
else:
    print("✅ requirements.txt lists every library you use")

⚠ Update requirements.txt so that it contains these lines:
    debugpy==1.8.22


## 6 · Write the output files (do not edit)

This section needs nothing from the cells above, only `team_model.pkl`. It writes `predictions.csv` from `predict_revenue` and `valuations.csv` from `auction_sheet`.

Here both output files are written from `sample_features.csv`, which has the same feature columns as the hidden files and `sample_id` as its row ID. **We re-run exactly these two cells with `TEST_FILE` and `AUCTION_FILE` set to the hidden test and auction files. Restart the kernel and confirm that these two cells run on their own.**

In [14]:
import gzip, pickle
import numpy as np
import pandas as pd

MODEL_FILE   = "team_model.pkl"
TEST_FILE    = "sample_features.csv"   # we use the hidden test file
AUCTION_FILE = "sample_features.csv"   # we use the hidden auction file

with gzip.open(MODEL_FILE, "rb") as f:
    saved_with = pickle.load(f)
    saved = pickle.load(f)
predict_revenue, auction_sheet = saved["predict_revenue"], saved["auction_sheet"]
print(f"Loaded {MODEL_FILE} (saved with Python {saved_with['python']}, scikit-learn {saved_with['scikit-learn']})")

Loaded team_model.pkl (saved with Python 3.13.15, scikit-learn 1.7.2)


In [15]:
ID_COLUMNS = ["test_id", "auction_id", "sample_id"]

def read_films(features_file):
    films = pd.read_csv(features_file)
    id_col = next(column for column in ID_COLUMNS if column in films.columns)
    return films[[id_col]], films.drop(columns=id_col)

# predictions.csv: one positive revenue forecast per test film (scored)
ids, films = read_films(TEST_FILE)
forecast = np.asarray(predict_revenue(films), dtype=float)
if forecast.ndim == 2 and forecast.shape[1] == 1:
    forecast = forecast[:, 0]
assert forecast.shape == (len(films),), f"expected one forecast per film, got shape {forecast.shape}"
assert np.isfinite(forecast).all() and (forecast > 0).all(), "every forecast must be a positive number"
ids.assign(predicted_revenue=forecast).to_csv("predictions.csv", index=False)
print(f"✅ predictions.csv: {len(forecast)} rows, median {np.median(forecast):,.0f} USD")

# valuations.csv: your auction sheet, with any columns you like (not scored)
ids, films = read_films(AUCTION_FILE)
sheet = pd.DataFrame(auction_sheet(films)).reset_index(drop=True)
assert len(sheet) == len(films), f"{len(sheet)} auction sheet rows for {len(films)} films"
assert sheet.columns.is_unique, "auction sheet column names must be unique"
assert not set(ID_COLUMNS) & set(sheet.columns), "section 6 adds the ID column; do not return it"
pd.concat([ids, sheet], axis=1).to_csv("valuations.csv", index=False)
print(f"✅ valuations.csv: {len(sheet)} rows, columns: {', '.join(map(str, sheet.columns))}")

✅ predictions.csv: 30 rows, median 35,255,801 USD
✅ valuations.csv: 30 rows, columns: valuation
